<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/appendix1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Accelerating Polars with the GPU

In [2]:
%pip install --upgrade cudf-polars-cu13

  Using cached cudf_polars_cu13-26.10.2-py3-none-any.whl.metadata (6.8 kB)
  Using cached cudf_streaming_cu13-26.10.2-cp311-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (1.1 kB)
  Using cached kvikio_cu13-26.10.0-cp311-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (4.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.8/865.8 kB 7.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 37.9 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.7/706.7 kB 39.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 951.2/951.2 kB 42.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 673.2/673.2 kB 38.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.6/549.6 kB 22.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.4/301.4 MB 44.0 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.5/28.5 MB 134.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 2.6 MB/s  0:00:01


In [1]:
import polars as pl
pl.__version__  # The book is built with Polars version 1.20.0

'1.35.2'

### Step 6: Test Your Installation

In [2]:
import polars as pl

pl.LazyFrame({"x": [1, 2, 3]}).collect(engine=pl.GPUEngine(raise_on_fail=True))

x
i64
1
2
3


## Using the Polars GPU Engine

### Configuration

### Unsupported Features

## Benchmarking the Polars GPU Engine

### Solutions

### Queries and Data

In [5]:
%%bash

mkdir -p sample_data/benchmark
curl -L -o "sample_data/benchmark/lineitem.parquet" \
  "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data/benchmark/lineitem.parquet"

curl -L -o "sample_data/benchmark/supplier.parquet" \
  "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data/benchmark/supplier.parquet"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 14.7M  100 14.7M    0     0  71.0M      0 --:--:-- --:--:-- --:--:-- 71.0M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 58866  100 58866    0     0   159k      0 --:--:-- --:--:-- --:--:--  159k


In [6]:
%%time
lineitem = pl.scan_parquet("sample_data/benchmark/lineitem.parquet")
supplier = pl.scan_parquet("sample_data/benchmark/supplier.parquet")

var1 = pl.date(1996, 1, 1)
var2 = pl.date(1996, 4, 1)

revenue = (
    lineitem.filter(pl.col("l_shipdate").is_between(var1, var2, closed="left"))
    .group_by("l_suppkey")
    .agg(
        (pl.col("l_extendedprice") * (1 - pl.col("l_discount")))
        .sum()
        .alias("total_revenue")
    )
    .select(pl.col("l_suppkey").alias("supplier_no"), pl.col("total_revenue"))
)

query_15 = (
    supplier.join(revenue, left_on="s_suppkey", right_on="supplier_no")
    .filter(pl.col("total_revenue") == pl.col("total_revenue").max())
    .with_columns(pl.col("total_revenue").round(2))
    .select("s_suppkey", "s_name", "s_address", "s_phone", "total_revenue")
    .sort("s_suppkey")
)

query_15.collect(engine="cpu")

CPU times: user 47.2 ms, sys: 10.5 ms, total: 57.6 ms
Wall time: 156 ms


s_suppkey,s_name,s_address,s_phone,total_revenue
i64,str,str,str,f64
677,"""Supplier#000000677""","""8mhrffG7D2WJBSQbOGstQ""","""23-290-639-3315""",1.6144e6


### Method

### Results and Discussion

#### Polars GPU engine versus CPU engine

#### Performance on different hardware

#### Polars GPU engine versus other packages

#### The effect of the Polars optimizer

### Conclusion

## The Future of Polars on the GPU

## Takeaways